In [ ]:
import typesense
import os
from dotenv import load_dotenv
load_dotenv(override=True)
client = typesense.Client({
  'nodes': [{
    'host': os.getenv('TYPESENSE_HOST_NAME'),  
    'port': '443',      
    'protocol': 'https'
  }],
  'api_key': os.getenv('TYPESENSE_API_KEY'),
  'connection_timeout_seconds': 2
})

In [2]:
books_schema={
    'name':'books',
    'fields':[
        {'name':'title', 'type':'string'},
        {'name':'authors', 'type':'string[]', 'facet':True},
        {'name':'publication_year', 'type':'int32', 'facet':True},
        {'name':'ratings_count', 'type':'int32'},
        {'name':'average_rating', 'type':'float'},
    ],
    'default_sorting_field':'ratings_count'
}

In [ ]:
try:
    client.collections['books'].delete()
except:
    pass
print(client.collections.create(books_schema))

In [7]:
with open('../data/json/books.jsonl', 'r', encoding='utf-8') as json1_file:
    data=json1_file.read()
    client.collections['books'].documents.import_(data)

In [ ]:
search_parameters={
    'q':"harry potter",
    'query_by':'title,authors',
    'filter_by':'publication_year:<1998',
    'sort_by':'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path
# from langchain_groq import ChatGroq

In [10]:
# llm = ChatGroq(
#     model="llama-3.3-70b-versatile",
#     api_key=os.getenv("GROQ_API_KEY"),
#     temperature=0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
# )

In [ ]:
#read all pdfs inside directory

def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} pdf files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader=PyMuPDFLoader(str(pdf_file))
            documents=loader.load()

            #adding more info to the metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"error: {e}")

    print(f"\nTotal document pages loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data/pdf")

In [ ]:
#split text into chunks
#sliding window chunking
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} document pages into {len(split_docs)} chunks")

    return split_docs

chunks=split_documents(all_pdf_documents)

In [13]:
embeddings=HuggingFaceEmbeddings()

In [14]:
docsearch=Typesense.from_documents(
    chunks,
    embeddings,
    typesense_client_params={
        'host': os.getenv('TYPESENSE_HOST_NAME'),  
        'port': '443',      
        'protocol': 'https',
        'typesense_api_key': os.getenv('TYPESENSE_API_KEY'),
        'typesense_collection_name':'pdf',
        'connection_timeout_seconds': 60
    },
)

In [ ]:
query="how is LLM used for code generation"
found_docs=docsearch.similarity_search(query)
print(found_docs[0].page_content)

In [ ]:
retriever=docsearch.as_retriever()
query="how is LLM used for code generation"
retriever.invoke(query)

In [28]:
from langchain_community.chat_models import ChatOllama
load_dotenv()
llm = ChatOllama(
    model="llama3.1",
    temperature=0,          
    base_url=os.getenv('OLLAMA_URL')
)

In [ ]:
def rag(query, retriever, llm, top_k=6):
    #retrieve the context
    results = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    
    prompt_template = """You are an expert Computer Science research assistant. 
Your task is to answer the question based STRICTLY on the provided context.

Guidelines:
1. **Be Precise**: Use technical terminology found in the context.
2. **Structure**: If the answer has multiple parts, use bullet points.
3. **No Hallucination**: If the context does not contain the answer, say "I cannot find the answer in the provided documents." Do not make up information.
4. **Synthesis**: If the answer is split across multiple chunks, combine them into a coherent explanation.

----------------
Context:
{context}
----------------

Question: {query}

Answer:"""
    final_prompt = prompt_template.format(context=context, query=query)
    response=llm.invoke(final_prompt)
    return response.content

In [ ]:
answer=rag("What are the characteristics of template-based code generation? Explain in detail.", retriever, llm)
answer

In [ ]:
from typing import Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm

    def query(self, question: str, top_k: int = 10, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

        if hasattr(self.retriever, 'search_kwargs'):
            self.retriever.search_kwargs['k'] = top_k
            
        print(f"Searching for: '{question}'...")
        results = self.retriever.invoke(question)
        
        if not results:
            return {"answer": "No relevant context found.", "sources": []}

        context = "\n\n".join([doc.page_content for doc in results])
        
        sources = []
        for doc in results:
    
            source_name = doc.metadata.get('source_file', doc.metadata.get('source', 'unknown'))
            page_num = doc.metadata.get('page', '?')
            sources.append({
                'source': source_name,
                'page': page_num,
                'preview': doc.page_content[:100].replace('\n', ' ') + '...'
            })

        prompt_template = """You are an expert research assistant. Answer the question based STRICTLY on the context provided.
If the answer is not in the context, say "I cannot find the answer in the documents."

Context:
{context}

Question: {question}

Answer:"""
        final_prompt = prompt_template.format(context=context, question=question)

        answer = ""
        
        if stream:
            print("\nStreaming Answer: ", end="", flush=True)
            
            for chunk in self.llm.stream(final_prompt):
                content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                print(content, end="", flush=True)
                answer += content
            print("\n") 
        else:
            response = self.llm.invoke(final_prompt)
            answer = response.content

        citation_lines = [f"[{i+1}] {src['source']} (Page {src['page']})" for i, src in enumerate(sources)]
        full_answer = answer + "\n\n**Sources:**\n" + "\n".join(citation_lines)

        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following text in one sentence:\n{answer}"
            summary_resp = self.llm.invoke(summary_prompt)
            summary = summary_resp.content

        return {
            'question': question,
            'answer': full_answer,
            'sources': sources,
            'summary': summary,
        }

adv_rag = AdvancedRAGPipeline(retriever, llm)

result = adv_rag.query(
    "What are the characteristics of template-based code generation?", 
    top_k=10, 
    stream=True,  
    summarize=True
)

print("\n-------------------")
print("Summary:", result['summary'])